# Inclusive-jet efficiency, matched rate, and fake rate

Estimate the inclusive-jet reconstruction efficiency, matched fraction, and fake fraction from the stored Lab-frame jet distributions. The two-dimensional ratios are shown directly in $(p_T^{jet}, \eta_{lab}^{jet})$; one-dimensional ratios are formed only after projecting numerator and denominator into the requested $p_T$ or $\eta$ interval. No yield or shape normalization is applied before division.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import math
import sys

PROJECT_ROOT = Path('/Users/gnigmat/work/cms/jetAnalysis')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    import ROOT
except ModuleNotFoundError:
    for path in (Path('/opt/homebrew/lib/python3.14/site-packages'),
                 Path('/opt/homebrew/Cellar/root/6.40.02_1/lib/root')):
        if path.exists() and str(path) not in sys.path:
            sys.path.insert(0, str(path))
    import ROOT

ROOT.gROOT.SetBatch(True)
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)
ROOT.TH1.AddDirectory(False)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import SINGLE_JET_PT_BINS
from hist_analysis.python.histogram_io import (
    load_histogram, resolve_combined_file, resolve_direction_file,
)
from hist_analysis.python.histogram_ops import ratio_to_nominal
from hist_analysis.python.projections import project_semantic_th2
from hist_analysis.python.root_style import (
    DEFAULT_PLOT_STYLE, draw_text_block, save_canvas, set_1d_style,
    set_2d_style, set_legend_style, set_pad_style, set_palette_style,
    style_single_panel_axes,
)

## Configuration

The defaults use the combined embedding file produced with the `jetId` stem. Projection intervals are half-open: $[low, high)$. `PT_BINS` controls the eta-dependent curves, while `ETA_RANGES` controls the pT-dependent curves. Binomial division is appropriate when the numerator is a subset of the denominator; because these are weighted MC histograms, the resulting uncertainties must still be interpreted as weighted-efficiency uncertainties rather than unweighted counting errors.

In [ ]:
GENERATOR = 'embedding'       # embedding or pythia
DIRECTION = 'combined'        # pgoing, Pbgoing, or combined
FILE_STEM = 'jetId'
PT_BINS = tuple(SINGLE_JET_PT_BINS)
ETA_RANGES = ((-2.4, 2.4),)
PT_DISPLAY_RANGE = (50.0, 500.0)
ETA_DISPLAY_RANGE = (-3.4, 3.4)
RATIO_OPTION = 'B'            # '' for independent errors, 'B' for binomial
RATE_RANGE = (0.0, 1.2)
MATCHED_FAKE_LOG_Y_RANGE = (1.0e-6, 2.0)
DRAW_GRID = True
SAVE_PNG = False
OUTPUT_DIR = PROJECT_ROOT / 'hist_analysis' / 'output' / 'jet_efficiency_fakes'

HISTOGRAM_KEYS = {
    'Reco matched': 'hRecoInclusiveJetPtEtaLabMatched',
    'Reco unmatched': 'hRecoInclusiveJetPtEtaLabUnmatched',
    'Reco inclusive': 'hRecoInclusiveJetPtEtaLab',
    'Ref matched': 'hRefInclusiveJetPtEtaLab',
    'Gen inclusive': 'hGenInclusiveJetPtEtaLab',
}
RATE_SPECS = {
    'Efficiency': ('Ref matched', 'Gen inclusive'),
    'Matched rate': ('Reco matched', 'Reco inclusive'),
    'Fake rate': ('Reco unmatched', 'Reco inclusive'),
}
STYLE_INDICES = {'Efficiency': 2, 'Matched rate': 0, 'Fake rate': 1}

def mc_file(generator, direction):
    if direction == 'combined':
        return resolve_combined_file(BASE_DIR, generator, FILE_STEM)
    return resolve_direction_file(BASE_DIR, generator, direction, FILE_STEM)

INPUT_FILE = mc_file(GENERATOR, DIRECTION)
if not INPUT_FILE.exists():
    raise FileNotFoundError(f'Missing configured ROOT file: {INPUT_FILE}')
INPUT_FILE

## Load and validate inputs

Every object must be a TH2 with exactly the same bin edges on both axes. Detached clones remain valid after each input file handle is closed.

In [ ]:
histograms_2d = {
    label: load_histogram(str(INPUT_FILE), key)
    for label, key in HISTOGRAM_KEYS.items()
}
not_th2 = [label for label, histogram in histograms_2d.items()
           if not histogram.InheritsFrom('TH2')]
if not_th2:
    raise TypeError(f'Expected TH2 inputs for: {not_th2}')

def axis_edges(axis):
    return tuple(axis.GetBinLowEdge(index)
                 for index in range(1, axis.GetNbins() + 2))

reference = histograms_2d['Reco inclusive']
reference_edges = (axis_edges(reference.GetXaxis()),
                   axis_edges(reference.GetYaxis()))
incompatible = []
for label, histogram in histograms_2d.items():
    edges = (axis_edges(histogram.GetXaxis()), axis_edges(histogram.GetYaxis()))
    if edges != reference_edges:
        incompatible.append(label)
if incompatible:
    raise ValueError(f'Incompatible 2D binning for: {incompatible}')

{label: (histogram.ClassName(), histogram.GetNbinsX(), histogram.GetNbinsY())
 for label, histogram in histograms_2d.items()}

## Two-dimensional rate maps

Each map is the direct bin-by-bin ratio of its stored numerator and denominator. Empty denominator bins are left at ROOT's zero result. The displayed color range is common to all three maps; histogram contents are not clipped.

In [ ]:
def ratio_2d(numerator, denominator, name):
    result = numerator.Clone(name)
    result.SetDirectory(0)
    result.Divide(numerator, denominator, 1.0, 1.0, RATIO_OPTION)
    return result

rates_2d = {
    label: ratio_2d(histograms_2d[numerator], histograms_2d[denominator],
                    f'h{label.replace(" ", "")}2D')
    for label, (numerator, denominator) in RATE_SPECS.items()
}

def output_tag(label):
    return label.lower().replace(' ', '_')

def draw_rate_map(histogram, label):
    plotted = histogram.Clone(f'{histogram.GetName()}_plot')
    plotted.SetDirectory(0)
    plotted.SetTitle('')
    plotted.GetXaxis().SetTitle('p_{T}^{jet} (GeV)')
    plotted.GetYaxis().SetTitle('#eta_{lab}^{jet}')
    plotted.GetZaxis().SetTitle(label)
    plotted.GetXaxis().SetRangeUser(*PT_DISPLAY_RANGE)
    plotted.GetYaxis().SetRangeUser(*ETA_DISPLAY_RANGE)
    plotted.SetMinimum(RATE_RANGE[0])
    plotted.SetMaximum(RATE_RANGE[1])
    set_2d_style(plotted)

    tag = output_tag(label)
    canvas = ROOT.TCanvas(
        f'c_{tag}_2d', '', DEFAULT_PLOT_STYLE.canvas_width,
        DEFAULT_PLOT_STYLE.canvas_height,
    )
    set_pad_style(canvas, grid_x=DRAW_GRID, grid_y=DRAW_GRID)
    canvas.SetRightMargin(DEFAULT_PLOT_STYLE.palette_right_margin)
    canvas.SetLogx(True)
    plotted.Draw('COLZ')
    annotations = draw_text_block(canvas, (
        GENERATOR.capitalize(), DIRECTION, 'Lab frame', label,
    ))
    canvas.Modified()
    canvas.Update()
    palette = set_palette_style(plotted)
    canvas.Modified()
    canvas.Update()
    save_canvas(canvas, OUTPUT_DIR / f'{tag}_2d.pdf', save_png=SAVE_PNG)
    canvas._rate_map_objects = [plotted, palette, *annotations]
    return canvas

map_canvases = {}
for label, histogram in rates_2d.items():
    map_canvases[label] = draw_rate_map(histogram, label)
    display(map_canvases[label])

## Projection and plotting helpers

Numerator and denominator are projected independently before division. Efficiency is shown alone, while matched and fake fractions are overlaid because they partition the inclusive reconstructed population.

In [ ]:
def rate_projection(rate_label, observable, selection_range, tag):
    numerator_label, denominator_label = RATE_SPECS[rate_label]
    numerator = project_semantic_th2(
        histograms_2d[numerator_label], observable, selection_range,
        name=f'h{output_tag(rate_label)}_{tag}_numerator',
    )
    denominator = project_semantic_th2(
        histograms_2d[denominator_label], observable, selection_range,
        name=f'h{output_tag(rate_label)}_{tag}_denominator',
    )
    ratio = ratio_to_nominal(
        numerator, denominator, name=f'h{output_tag(rate_label)}_{tag}',
        option=RATIO_OPTION,
    )
    return {'numerator': numerator, 'denominator': denominator, 'rate': ratio}

def prepare_rate_axes(histogram, x_title):
    histogram.SetTitle('')
    histogram.GetXaxis().SetTitle(x_title)
    histogram.GetYaxis().SetTitle('Rate')
    histogram.SetMinimum(RATE_RANGE[0])
    histogram.SetMaximum(RATE_RANGE[1])
    style_single_panel_axes(histogram)

def draw_single_rate(histogram, label, x_title, x_range, annotations, tag, log_x=False):
    plotted = histogram.Clone(f'{histogram.GetName()}_plot')
    plotted.SetDirectory(0)
    set_1d_style(plotted, STYLE_INDICES[label])
    prepare_rate_axes(plotted, x_title)
    plotted.GetXaxis().SetRangeUser(*x_range)
    canvas = ROOT.TCanvas(
        f'c_{tag}', '', DEFAULT_PLOT_STYLE.canvas_width,
        DEFAULT_PLOT_STYLE.canvas_height,
    )
    set_pad_style(canvas, grid_x=DRAW_GRID, grid_y=DRAW_GRID)
    canvas.SetLogx(log_x)
    plotted.Draw('E1')
    labels = draw_text_block(canvas, (*annotations, label))
    canvas.Modified()
    canvas.Update()
    save_canvas(canvas, OUTPUT_DIR / f'{tag}.pdf', save_png=SAVE_PNG)
    canvas._single_rate_objects = [plotted, *labels]
    return canvas

def draw_matched_fake(rates, x_title, x_range, annotations, tag, log_x=False):
    plotted = {}
    for label in ('Matched rate', 'Fake rate'):
        histogram = rates[label].Clone(f'{rates[label].GetName()}_plot')
        histogram.SetDirectory(0)
        set_1d_style(histogram, STYLE_INDICES[label])
        prepare_rate_axes(histogram, x_title)
        histogram.SetMinimum(MATCHED_FAKE_LOG_Y_RANGE[0])
        histogram.SetMaximum(MATCHED_FAKE_LOG_Y_RANGE[1])
        histogram.GetXaxis().SetRangeUser(*x_range)
        plotted[label] = histogram
    canvas = ROOT.TCanvas(
        f'c_{tag}', '', DEFAULT_PLOT_STYLE.canvas_width,
        DEFAULT_PLOT_STYLE.canvas_height,
    )
    set_pad_style(canvas, grid_x=DRAW_GRID, grid_y=DRAW_GRID)
    canvas.SetLogx(log_x)
    canvas.SetLogy(True)
    legend = ROOT.TLegend(0.66, 0.76, 0.88, 0.88)
    set_legend_style(legend)
    for index, (label, histogram) in enumerate(plotted.items()):
        histogram.Draw('E1' if index == 0 else 'E1 SAME')
        legend.AddEntry(histogram, label, 'p')
    legend.Draw()
    labels = draw_text_block(canvas, annotations)
    canvas.Modified()
    canvas.Update()
    save_canvas(canvas, OUTPUT_DIR / f'{tag}.pdf', save_png=SAVE_PNG)
    canvas._matched_fake_objects = [*plotted.values(), legend, *labels]
    return canvas

## Rates versus $\eta_{lab}^{jet}$ in fixed pT intervals

In [ ]:
eta_results = {}
for low, high in PT_BINS:
    interval_tag = f'pt_{low:g}_{high:g}'.replace('.', 'p')
    rates = {
        label: rate_projection(label, 'eta', (low, high), interval_tag)
        for label in RATE_SPECS
    }
    annotations = (
        GENERATOR.capitalize(), DIRECTION, 'Lab frame',
        f'{low:g} < p_{{T}}^{{jet}} < {high:g} GeV',
    )
    efficiency_canvas = draw_single_rate(
        rates['Efficiency']['rate'], 'Efficiency', '#eta_{lab}^{jet}',
        ETA_DISPLAY_RANGE, annotations, f'efficiency_eta_{interval_tag}',
    )
    matched_fake_canvas = draw_matched_fake(
        {label: result['rate'] for label, result in rates.items()},
        '#eta_{lab}^{jet}', ETA_DISPLAY_RANGE, annotations,
        f'matched_fake_eta_{interval_tag}',
    )
    eta_results[(low, high)] = {
        'rates': rates, 'efficiency_canvas': efficiency_canvas,
        'matched_fake_canvas': matched_fake_canvas,
    }
    display(efficiency_canvas)
    display(matched_fake_canvas)

## Rates versus $p_T^{jet}$ in fixed eta intervals

In [ ]:
pt_results = {}
for low, high in ETA_RANGES:
    interval_tag = f'eta_{low:g}_{high:g}'.replace('-', 'm').replace('.', 'p')
    rates = {
        label: rate_projection(label, 'pt', (low, high), interval_tag)
        for label in RATE_SPECS
    }
    annotations = (
        GENERATOR.capitalize(), DIRECTION, 'Lab frame',
        f'{low:g} < #eta_{{lab}}^{{jet}} < {high:g}',
    )
    efficiency_canvas = draw_single_rate(
        rates['Efficiency']['rate'], 'Efficiency', 'p_{T}^{jet} (GeV)',
        PT_DISPLAY_RANGE, annotations, f'efficiency_pt_{interval_tag}',
        log_x=True,
    )
    matched_fake_canvas = draw_matched_fake(
        {label: result['rate'] for label, result in rates.items()},
        'p_{T}^{jet} (GeV)', PT_DISPLAY_RANGE, annotations,
        f'matched_fake_pt_{interval_tag}', log_x=True,
    )
    pt_results[(low, high)] = {
        'rates': rates, 'efficiency_canvas': efficiency_canvas,
        'matched_fake_canvas': matched_fake_canvas,
    }
    display(efficiency_canvas)
    display(matched_fake_canvas)

## Numerical audit

The summary reports stored weighted yields before division, zero-denominator bins, the full finite rate range, and the largest deviation of matched plus fake from unity where the inclusive-reco denominator is nonzero. Values outside the displayed `[0,1]` plotting range remain visible here.

In [ ]:
def audit_projection(kind, selection_range, results):
    print(f'{kind} interval {selection_range}')
    for label, result in results.items():
        numerator = result['numerator']
        denominator = result['denominator']
        rate = result['rate']
        regular_bins = range(1, denominator.GetNbinsX() + 1)
        zero_bins = sum(denominator.GetBinContent(index) == 0.0
                        for index in regular_bins)
        finite_rates = [rate.GetBinContent(index) for index in regular_bins
                        if math.isfinite(rate.GetBinContent(index))]
        rate_min = min(finite_rates, default=float('nan'))
        rate_max = max(finite_rates, default=float('nan'))
        print(
            f'  {label:12s}: numerator={numerator.Integral():.8g}, '
            f'denominator={denominator.Integral():.8g}, '
            f'zero denominator bins={zero_bins}, '
            f'range=[{rate_min:.6g}, {rate_max:.6g}]'
        )
    matched = results['Matched rate']['rate']
    fake = results['Fake rate']['rate']
    denominator = results['Matched rate']['denominator']
    deviations = [
        abs(matched.GetBinContent(index) + fake.GetBinContent(index) - 1.0)
        for index in range(1, denominator.GetNbinsX() + 1)
        if denominator.GetBinContent(index) != 0.0
    ]
    print(f'  max |matched + fake - 1| = {max(deviations, default=0.0):.3g}')

for interval, result in eta_results.items():
    audit_projection('pT', interval, result['rates'])
for interval, result in pt_results.items():
    audit_projection('eta', interval, result['rates'])

reco_partition = histograms_2d['Reco matched'].Clone('hRecoPartitionCheck')
reco_partition.SetDirectory(0)
reco_partition.Add(histograms_2d['Reco unmatched'])
max_partition_difference = max(
    abs(reco_partition.GetBinContent(xbin, ybin)
        - histograms_2d['Reco inclusive'].GetBinContent(xbin, ybin))
    for xbin in range(1, reco_partition.GetNbinsX() + 1)
    for ybin in range(1, reco_partition.GetNbinsY() + 1)
)
print(f'2D max |matched + unmatched - inclusive reco| = '
      f'{max_partition_difference:.3g}')